# TC_10: CNN clean baseline
Full diagnostic pipeline on clean image data using simple CNN model.
UQ: Deep Ensemble, MC Dropout
XAI: Grad-CAM

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch


sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_image_data
from src.data_diagnostics.quality_checks import check_image_quality
from src.utils.visualization import plot_class_distribution, plot_correlation, plot_feature_boxplots
from src.utils.performance_tracker import PerformanceTracker
from src.models.image_models import train_simple_cnn, evaluate_image_model
from src.inference_diagnostics.uncertainty import mc_dropout_prediction_img, deep_ensemble_prediction_img
from src.inference_diagnostics.explainability import gradcam_img, collect_gradcam_feedback
from src.inference_diagnostics.flagging import assign_flags, evaluate_flags
from src.utils.visualization import plot_uncertainty_distribution, plot_flag_distribution, plot_gradcam_sample
from src.utils.report_generator import generate_report,save_report
from src.utils.config_loader import load_config, get_image_config, save_threshold
from src.utils.per_sample_logger import save_per_sample, build_experiment_id
from src.inference_diagnostics.explainability import collect_gradcam_feedback_resumable



config = load_config()
tracker = PerformanceTracker()

dataset_short = get_image_config(config)['short_name']
model_name = 'simple_cnn'
test_case = 'TC10'
review_count = config['stage3_inference']['explainability']['gradcam']['max_samples_to_explain']
experiment_id = build_experiment_id(dataset_short, model_name, test_case)
print(f"Experiment: {experiment_id}, reviewing {review_count} samples")

Experiment: cifar10_simple_cnn_TC10, reviewing 100 samples


### Load image data and EDA


In [2]:
train_set, test_set = load_image_data(config)
report = check_image_quality(train_set, config)
plot_class_distribution(report, "CNN CIFAR-10 Baseline", config)

Loading dataset.
Dataset: CIFAR-10
Loaded: 50000 train and 10000 test
Image size: 32, Channels: 3
Classes: 10
Class names: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
EDA started for image data.
Total images: 50000
Classes: 10
Distribution: {0: 5000, 1: 5000, 2: 5000, 3: 5000, 4: 5000, 5: 5000, 6: 5000, 7: 5000, 8: 5000, 9: 5000}
Imbalance ratio: 1.0
EDA completed for CIFAR-10
Saved: results/figures\class_distribution_cnn_cifar-10_baseline.png


### Train CNN baseline

In [3]:
tracker.start_performance_track()
cnn_model = train_simple_cnn(train_set, config)
tracker.stop_performance_track("CNN training")

tracker.start_performance_track()
cnn_accuracy, cnn_prediction, cnn_report = evaluate_image_model(cnn_model, test_set, config)
tracker.stop_performance_track("CNN evaluation")

Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
CNN training: Time:437.69s, Memory:701.11MB
 Image model evaluation started.
{'0': {'precision': 0.8331550802139037, 'recall': 0.779, 'f1-score': 0.8051679586563307, 'support': 1000.0}, '1': {'precision': 0.8702213279678068, 'recall': 0.865, 'f1-score': 0.8676028084252758, 'support': 1000.0}, '2': {'precision': 0.6595330739299611, 'recall': 0.678, 'f1-score': 0.6686390532544378, 'support': 1000.0}, '3': {'precision': 0.6050884955752213, 'recall': 0.547, 'f1-score': 0.5745798319327731, 'support': 1000.0}, '4': {'precision': 0.7635294117647059, 'recall': 0.649, 'f1-score': 0.7016216216216217, 'support': 1000.0}, '5': {'precision': 0.6407129455909943, 'recall': 0.683, 'f1-score': 0.6611810261374637, 'support': 1000.0}, '6': {'precision': 0.7448630136986302, 'recall': 0.87, 'f1-score': 0.8025830258302583, 'support': 1000.0}, '7': {'precision': 0.8162217659137577, 'recall': 0.795, 'f1-

{'operation': 'CNN evaluation', 'time_seconds': 3.71, 'memory_usage': 9.3}

### Uncertainty Quantification (MC Dropout + Deep Ensemble)

In [4]:
# MC Dropout
tracker.start_performance_track()
mc_means_probabilities, mc_uncertainty = mc_dropout_prediction_img(cnn_model, test_set, config)
tracker.stop_performance_track("CNN MC Dropout")

mc_threshold = round(float(np.percentile(mc_uncertainty, 90)), 4)
save_threshold(config, dataset_short, model_name, 'mc', mc_threshold)
print(f"MC Dropout uncertainty status:")
print(f"Mean: {mc_uncertainty.mean()}")
print(f"Max: {mc_uncertainty.max()}")
print(f" 90th percentile (threshold): {mc_threshold}")

plot_uncertainty_distribution(mc_uncertainty, "CNN MC Dropout", mc_threshold, config)

MC Dropout started.
Pass 10/50 done.
Pass 20/50 done.
Pass 30/50 done.
Pass 40/50 done.
Pass 50/50 done.
MC Dropout finished for images.
CNN MC Dropout: Time:203.50s, Memory:20.26MB
Threshold saved: cifar10_simple_cnn_mc = 0.0827
MC Dropout uncertainty status:
Mean: 0.030484048649668694
Max: 0.154929980635643
 90th percentile (threshold): 0.0827
Saved: results/figures\uncertainty_distribution_cnn_mc_dropout.png


In [5]:
# Deep Ensemble
tracker.start_performance_track()
de_means_probabilities, de_uncertainty = deep_ensemble_prediction_img(train_simple_cnn, config, test_set, train_set  )
tracker.stop_performance_track("CNN Deep Ensemble prediction")

de_threshold = round(float(np.percentile(de_uncertainty, 90)), 4)
save_threshold(config, dataset_short, model_name, 'de', de_threshold)
print(f"Deep Ensembl uncertainty status:")
print(f"Mean: {de_uncertainty.mean()}")
print(f"Max: {de_uncertainty.max()}")
print(f" 90th percentile (threshold): {de_threshold}")

plot_uncertainty_distribution(de_uncertainty, "CNN Deep Ensemble", de_threshold, config)

Deep Ensemble started for images.
Training model with seed 0
Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
Training model with seed 1
Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
Training model with seed 2
Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
Training model with seed 3
Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
Training model with seed 4
Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
Deep Ensemble finished for images.
CNN Deep Ensemble prediction: Time:2510.76s, Memory:0.00MB
Threshold saved: cifar10_simple_cnn_de = 0.1062
Deep Ensembl uncertainty status:
Mean: 0.041962992399930954
Max: 0.19042234122753143
 90th percentile (threshold): 0.1062
Saved: results/figures\uncertain

### Explainability (Grad-CAM and Human feedback)

In [6]:
tracker.start_performance_track()
heatmaps = gradcam_img(cnn_model, test_set, config)
tracker.stop_performance_track("CNN Grad-CAM")

# Show grid for expert review
plot_gradcam_sample(test_set, heatmaps, review_count, config, "CNN CIFAR-10 Baseline", save = True)


GradCam started.
GradCam finished. Explained: 50 / 100 samples.
GradCam finished. Explained: 100 / 100 samples.
GradCam finished. Explained: 100 samples.
CNN Grad-CAM: Time:0.41s, Memory:18.09MB
Saved: results/figures\gradcam_sample_cnn_cifar-10_baseline.png


### Collect human feedback

In [7]:
# consistency_scores, correct, incorrect, partial = collect_gradcam_feedback(review_count)
# print(f"Human feedback: correct: {correct}, incorrect: {incorrect}, partial: {partial}")

consistency_scores, correct, incorrect, partial = collect_gradcam_feedback_resumable(
    review_count, config, experiment_id, batch_size=10
)

Resuming cifar10_simple_cnn_TC10: 10/100 already done.
Scoring: 1 = Correct, 2 = Partial, 3 = Incorrect
Review in batches of 10. Type 'stop' to pause and resume later.

  Saved. Progress: 20/100
  Saved. Progress: 30/100
  Saved. Progress: 40/100
  Saved. Progress: 50/100
  Saved. Progress: 60/100
  Saved. Progress: 70/100
  Saved. Progress: 80/100
  Saved. Progress: 90/100
  Saved. Progress: 100/100
Correct: 10, Incorrect: 81, Partial: 9


### Flagging (UQ + Grad-CAM feedback)

In [8]:
# Get true  labels for review_count samples
y_true_reviewed  = np.array([test_set[i][1] for i in range(review_count)])

# MC Dropout
mc_y_pred_reviewed = mc_means_probabilities[:review_count].argmax(axis = 1)
mc_flags_reviewed = assign_flags(mc_uncertainty[:review_count], consistency_scores, mc_threshold, 0.5)
mc_flag_result_reviewed = evaluate_flags(mc_flags_reviewed, mc_y_pred_reviewed, y_true_reviewed)
plot_flag_distribution(mc_flags_reviewed, "CNN MC + Grad-CAM Reviewed", config)

# Deep Ensemble
de_y_pred_reviewed = de_means_probabilities[:review_count].argmax(axis = 1)
de_flags_reviewed = assign_flags(de_uncertainty[:review_count], consistency_scores, de_threshold, 0.5)
de_flag_result_reviewed = evaluate_flags(de_flags_reviewed, de_y_pred_reviewed, y_true_reviewed)
plot_flag_distribution(de_flags_reviewed, "CNN DE + Grad-CAM Reviewed", config)

RED: Count: 5 Accuracy:60.0%
YELLOW: Count: 82 Accuracy:75.6%
GREEN: Count: 13 Accuracy:53.8%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_cnn_mc_+_grad-cam_reviewed.png
RED: Count: 7 Accuracy:28.6%
YELLOW: Count: 80 Accuracy:86.2%
GREEN: Count: 13 Accuracy:76.9%
Flagging system is working
Saved: results/figures\flagging_distribution_cnn_de_+_grad-cam_reviewed.png


### Full flagging with constant consistency for all sample

In [9]:
# Full test set flagging (UQ only, no human review)
fully_y_true = np.array([test_set[i][1] for i in range(len(test_set))])
fake_consistency = np.ones(len(mc_uncertainty))

# MC only
full_mc_y_prediction = mc_means_probabilities.argmax(axis = 1)
mc_flags_full = assign_flags(mc_uncertainty, fake_consistency, mc_threshold, 0.5)
mc_full_result = evaluate_flags(mc_flags_full, full_mc_y_prediction, fully_y_true)
plot_flag_distribution(mc_flags_full, "CNN MC UQ Only Full", config)

# Deep Ensemble only
full_de_y_prediction = de_means_probabilities.argmax(axis = 1)
de_flags_full = assign_flags(de_uncertainty, fake_consistency, de_threshold, 0.5)
de_full_result = evaluate_flags(de_flags_full, full_de_y_prediction, fully_y_true)
plot_flag_distribution(de_flags_full, "CNN DE UQ Only Full", config)

RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 999 Accuracy:37.6%
GREEN: Count: 9001 Accuracy:80.4%
Flagging system is working
Saved: results/figures\flagging_distribution_cnn_mc_uq_only_full.png
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 1000 Accuracy:44.6%
GREEN: Count: 9000 Accuracy:86.1%
Flagging system is working
Saved: results/figures\flagging_distribution_cnn_de_uq_only_full.png


In [10]:
# Reviewed-sample flagging (UQ only, no human review)
fake_consistency_reviewed = np.ones(review_count)

# MC only
mc_flags_uq_only = assign_flags(mc_uncertainty[:review_count], fake_consistency_reviewed, mc_threshold, 0.5)
mc_20_result = evaluate_flags(mc_flags_uq_only, mc_y_pred_reviewed, y_true_reviewed)
plot_flag_distribution(mc_flags_uq_only, "CNN MC UQ Only reviewed", config)

# Deep Ensemble only
de_flags_uq_only = assign_flags(de_uncertainty[:review_count], fake_consistency_reviewed, de_threshold, 0.5)
de_20_result = evaluate_flags(de_flags_uq_only, de_y_pred_reviewed, y_true_reviewed)
plot_flag_distribution(de_flags_uq_only, "CNN DE UQ Only reviewed", config)

RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 11 Accuracy:36.4%
GREEN: Count: 89 Accuracy:76.4%
Flagging system is working
Saved: results/figures\flagging_distribution_cnn_mc_uq_only_reviewed.png
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 13 Accuracy:46.2%
GREEN: Count: 87 Accuracy:86.2%
Flagging system is working
Saved: results/figures\flagging_distribution_cnn_de_uq_only_reviewed.png


In [11]:
save_per_sample(
    config,
    experiment_id,
    y_true=y_true_reviewed,
    y_pred=mc_y_pred_reviewed,
    mc_uncertainty=mc_uncertainty[:review_count],
    de_uncertainty=de_uncertainty[:review_count],
    consistency=consistency_scores,
)

Per sample results saved: results/per_sample\cifar10_simple_cnn_TC10.csv (100 rows)


,sample_id,y_true,y_pred,correct,mc_uncertainty,de_uncertainty,consistency
0,0,0,0,1,6.287724e-02,0.023252,0.0
1,1,0,0,1,1.098539e-04,0.048882,0.5
2,2,0,2,0,9.314047e-03,0.090277,0.0
3,3,0,0,1,3.692884e-04,0.029979,0.5
4,4,0,0,1,2.707704e-02,0.003077,0.0
...,...,...,...,...,...,...,...
95,95,0,2,0,3.076842e-02,0.085862,0.0
96,96,0,8,0,3.078040e-02,0.085110,0.0
97,97,0,0,1,3.534420e-04,0.000235,0.0
98,98,0,0,1,2.817998e-07,0.025265,0.0


### Save golden baseline report

In [12]:
cnn_baseline = {
    'model': 'Simple CNN',
    'accuracy': round(cnn_accuracy, 4),
    'classification_report': cnn_report,
    'mc_uncertainty': {
        'mean': round(float(mc_uncertainty.mean()), 8),
        'max': round(float(mc_uncertainty.max()), 8),
        'threshold': mc_threshold
    },
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold
    },
    'gradcam_feedback':{
        'correct': int(correct),
        'incorrect': int(incorrect),
        'partial': int(partial)
    },
    'flagging_mc': mc_flag_result_reviewed,
    'flagging_de': de_flag_result_reviewed,
    'flagging_mc_20_mc_only': mc_20_result,
    'flagging_mc_20_de_only': de_20_result,
    'flagging_mc_full': mc_full_result,
    'flagging_de_full': de_full_result,
    'mc_vs_mc_plus_xai': round(mc_flag_result_reviewed['GREEN']['accuracy'] - mc_20_result['GREEN']['accuracy'], 4),
    'de_vs_de_plus_xai': round(de_flag_result_reviewed['GREEN']['accuracy'] - de_20_result['GREEN']['accuracy'], 4),
    'performance': tracker.get_results()
}

report_output = generate_report(config, f"{get_image_config(config)['name']} - CNN golden baseline", stage2_result=cnn_baseline)
save_report(report_output, f'{dataset_short.upper()}_TC_10_Simple_CNN_Golden_Baseline.json', config)

Generating report.
Diagnostic report generated.
Saving report.
